# Air Quality Forecasting - Beijing PM2.5

Predicting daily PM2.5 levels in Beijing using historical air quality and weather data. I compare three approaches: ARIMA (classical statistical), Prophet (Facebook's decomposition model), and Random Forest with lag features (ML approach). Data is from 12 monitoring stations across Beijing (2013-2017). I use one station (Aotizhongxin) and aggregate hourly readings to daily averages.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

## 2. Load and Explore

In [ ]:
df = pd.read_csv('../data/beijing_air_quality.csv')
print('Shape:', df.shape)
print('Stations:', df['station'].nunique())
print('Years:', sorted(df['year'].unique()))
df.head()

In [ ]:
# Missing values per column
print('Missing values:')
print(df.isna().sum()[df.isna().sum() > 0])

## 3. Data Preparation

Filtering to one station (Aotizhongxin) to keep the analysis focused. Aggregating hourly readings to daily averages - this reduces noise and is a more appropriate resolution for forecasting since daily patterns are more stable than hour-to-hour spikes. Missing hourly values are forward-filled before aggregation (sensor outages typically last a few hours so the previous reading is a reasonable fill).

In [ ]:
# Filter to one station
station = df[df['station'] == 'Aotizhongxin'].copy()

# Build datetime index
station['date'] = pd.to_datetime(station[['year', 'month', 'day']])

# Forward fill missing values within each day before aggregating
station = station.sort_values('date')
station['PM2.5'] = station['PM2.5'].fillna(method='ffill')
station['TEMP'] = station['TEMP'].fillna(method='ffill')
station['PRES'] = station['PRES'].fillna(method='ffill')
station['WSPM'] = station['WSPM'].fillna(method='ffill')

# Aggregate to daily averages
daily = station.groupby('date').agg(
    PM25=('PM2.5', 'mean'),
    TEMP=('TEMP', 'mean'),
    PRES=('PRES', 'mean'),
    RAIN=('RAIN', 'sum'),
    WSPM=('WSPM', 'mean'),
).reset_index()

# Drop any remaining NaN days
daily = daily.dropna(subset=['PM25']).reset_index(drop=True)

print(f'Daily records: {len(daily)}')
print(f'Date range: {daily["date"].min()} to {daily["date"].max()}')
daily.head()

In [ ]:
# Plot PM2.5 over the full period
plt.figure(figsize=(14, 4))
plt.plot(daily['date'], daily['PM25'], linewidth=0.8, alpha=0.8)
plt.xlabel('Date')
plt.ylabel('PM2.5')
plt.title('Daily PM2.5 - Aotizhongxin station (2013-2017)')
plt.tight_layout()
plt.show()

print('PM2.5 stats:')
print(daily['PM25'].describe().round(2))

In [ ]:
# Distribution of PM2.5
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(daily['PM25'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_xlabel('PM2.5')
axes[0].set_title('PM2.5 distribution (right skewed)')

# Monthly average to see seasonality
daily['month'] = daily['date'].dt.month
monthly_avg = daily.groupby('month')['PM25'].mean()
axes[1].bar(monthly_avg.index, monthly_avg.values, color='steelblue')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Average PM2.5')
axes[1].set_title('Seasonal pattern - winter months worse')
axes[1].set_xticks(range(1, 13))
plt.tight_layout()
plt.show()

## 4. Feature Engineering

Converting the time series into a supervised ML problem by creating lag features and rolling statistics. Each row becomes: given PM2.5 and weather from the past N days, what is today's PM2.5?

Features I create:
- **Lag features**: PM2.5 from 1, 3, 7, 14, 30 days ago
- **Rolling averages**: 7-day and 30-day rolling mean of PM2.5
- **Rolling std**: 7-day rolling standard deviation (captures volatility)
- **Time features**: day of week, month, season (winter = high pollution in Beijing)
- **Weather features**: temperature, pressure, rainfall, wind speed

In [ ]:
daily['dayofweek'] = daily['date'].dt.dayofweek
daily['season'] = daily['date'].dt.month.map(
    {12: 0, 1: 0, 2: 0, 3: 1, 4: 1, 5: 1, 6: 2, 7: 2, 8: 2, 9: 3, 10: 3, 11: 3}
)

# Lag features
for lag in [1, 3, 7, 14, 30]:
    daily[f'PM25_lag{lag}'] = daily['PM25'].shift(lag)

# Rolling features
daily['PM25_roll7'] = daily['PM25'].shift(1).rolling(7).mean()
daily['PM25_roll30'] = daily['PM25'].shift(1).rolling(30).mean()
daily['PM25_std7'] = daily['PM25'].shift(1).rolling(7).std()

# Drop rows where lags are NaN (first 30 days)
daily = daily.dropna().reset_index(drop=True)

print(f'Records after feature engineering: {len(daily)}')
print(f'Features: {[c for c in daily.columns if c not in ["date", "PM25", "month"]]}')
daily.head(3)

## 5. Train/Test Split

For time series data the split must be done by time, not randomly. Using the first 80% of days for training and the last 20% for testing. This simulates a real forecasting scenario where we train on the past and predict the future.

In [ ]:
# Time based split - no shuffling
split_idx = int(len(daily) * 0.8)
split_date = daily['date'].iloc[split_idx]
print(f'Training: up to {split_date.date()}')
print(f'Test: {split_date.date()} onwards')

feature_cols = [c for c in daily.columns
                if c not in ['date', 'PM25', 'month']]

X_train = daily.iloc[:split_idx][feature_cols]
X_test = daily.iloc[split_idx:][feature_cols]
y_train = daily.iloc[:split_idx]['PM25']
y_test = daily.iloc[split_idx:]['PM25']
dates_test = daily.iloc[split_idx:]['date']

print(f'Train size: {len(X_train)}, Test size: {len(X_test)}')

# PM2.5 series for ARIMA and Prophet (need full series)
pm25_series = daily.set_index('date')['PM25']

## 6. Models

Comparing three approaches:

1. **ARIMA** - classical statistical model that uses autoregression and moving averages on the series itself
2. **Prophet** - Facebook's decomposition model that explicitly models trend, seasonality, and holidays
3. **Random Forest** - ML model trained on the engineered lag/weather features above

### 6.1 ARIMA

ARIMA(p, d, q): p = autoregressive order (how many past values), d = differencing (to make stationary), q = moving average order. Using p=5, d=1, q=0 as a starting point - differencing once removes the trend, and 5 lags captures a work-week cycle.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Use only the training portion of the series
train_series = pm25_series.iloc[:split_idx]
test_series = pm25_series.iloc[split_idx:]

arima_model = ARIMA(train_series, order=(5, 1, 0))
arima_fit = arima_model.fit()

# Forecast for the test period
arima_pred = arima_fit.forecast(steps=len(test_series))
arima_pred.index = test_series.index

arima_rmse = np.sqrt(mean_squared_error(test_series, arima_pred))
arima_mae = mean_absolute_error(test_series, arima_pred)
print(f'ARIMA - RMSE: {arima_rmse:.2f}, MAE: {arima_mae:.2f}')

### 6.2 Prophet

Prophet decomposes the series into trend + yearly seasonality + weekly seasonality + holidays. It handles missing values and outliers well and requires minimal tuning. The input format needs columns named 'ds' (date) and 'y' (target).

In [ ]:
from prophet import Prophet

# Prophet needs ds and y columns
prophet_train = pd.DataFrame({
    'ds': pm25_series.index[:split_idx],
    'y': pm25_series.values[:split_idx]
})

prophet_model = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
prophet_model.fit(prophet_train)

# Forecast
future = pd.DataFrame({'ds': pm25_series.index[split_idx:]})
prophet_forecast = prophet_model.predict(future)
prophet_pred = prophet_forecast['yhat'].values
prophet_pred = np.clip(prophet_pred, 0, None)

prophet_rmse = np.sqrt(mean_squared_error(test_series.values, prophet_pred))
prophet_mae = mean_absolute_error(test_series.values, prophet_pred)
print(f'Prophet - RMSE: {prophet_rmse:.2f}, MAE: {prophet_mae:.2f}')

### 6.3 Random Forest with Lag Features

Treats the forecasting problem as a standard regression task using the engineered features. The key advantage is it can use weather data and multiple lags simultaneously, which ARIMA and Prophet can't do as naturally.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_mae = mean_absolute_error(y_test, rf_pred)
print(f'Random Forest - RMSE: {rf_rmse:.2f}, MAE: {rf_mae:.2f}')

## 7. Model Comparison

In [ ]:
# Summary table
results = pd.DataFrame([
    {'Model': 'ARIMA(5,1,0)', 'RMSE': round(arima_rmse, 2), 'MAE': round(arima_mae, 2)},
    {'Model': 'Prophet', 'RMSE': round(prophet_rmse, 2), 'MAE': round(prophet_mae, 2)},
    {'Model': 'Random Forest', 'RMSE': round(rf_rmse, 2), 'MAE': round(rf_mae, 2)},
])
results

In [ ]:
# Predicted vs actual for all three models
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

models_preds = [
    ('ARIMA', arima_pred.values),
    ('Prophet', prophet_pred),
    ('Random Forest', rf_pred),
]

for ax, (name, pred) in zip(axes, models_preds):
    ax.plot(dates_test.values, test_series.values, label='Actual', linewidth=1, alpha=0.8)
    ax.plot(dates_test.values, pred, label=f'{name} predicted', linewidth=1, alpha=0.8)
    ax.set_ylabel('PM2.5')
    ax.set_title(f'{name} - Predicted vs Actual')
    ax.legend()

plt.xlabel('Date')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance from Random Forest
feat_imp = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=True).tail(10)

plt.figure(figsize=(8, 5))
plt.barh(feat_imp['feature'], feat_imp['importance'], color='steelblue')
plt.xlabel('Importance')
plt.title('Random Forest - Top 10 feature importances')
plt.tight_layout()
plt.show()

## 8. Conclusion

All three models were trained on daily PM2.5 readings from the Aotizhongxin station (2013 to early 2017) and evaluated on the final 20% of dates.

Random Forest outperformed both ARIMA and Prophet on RMSE and MAE. The lag features (especially PM2.5 from the previous 1 and 7 days) and the 7-day rolling average were the most important predictors according to the feature importance plot. This makes sense - air quality is strongly autocorrelated, yesterday's reading is a strong predictor of today's.

ARIMA captured the broad trend but struggled with the sharp pollution spikes that are common in Beijing winter months. Prophet captured seasonality well (correctly identifying winter months as higher pollution) but oversmoothed the daily variation.

The seasonal pattern is clear in the data - PM2.5 peaks in December to February and drops in summer months, consistent with coal heating and temperature inversion effects in Beijing.

Limitations:

- Analysis uses one station only. The 12 stations have different pollution profiles and a spatial model would capture more.
- ARIMA order (5,1,0) was chosen manually. Auto-ARIMA could find a better order.
- Prophet and ARIMA don't use the weather features, putting them at a disadvantage vs Random Forest.

Future work:

- Try gradient boosting (XGBoost) with the same features, likely to improve on Random Forest.
- Add Prophet regressors for weather variables so it can compete more fairly.
- Build a multi-station spatial model to capture pollution spillover between areas.
- Try LSTM for sequence modeling since it can learn long range dependencies automatically.